# MS_C2 — Multi-Station CNN-LSTM (Perfect Forecast Full (HYSPLIT + MET at t+h))

Trains one Bidirectional CNN-LSTM (MIMO, 24 horizons) per station in `STATIONS_TO_RUN`.
Uses `weather_mode="perfect_forecast_full"` (MET_COLS + HYSPLIT shifted to t+24 in each sequence window).

**Checkpoint:** skips a station if `outputs/{station}/results/C2_metrics.csv` already exists.
A kernel restart resumes from the last incomplete station.

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import time
import numpy as np
import pandas as pd

import src.config as cfg
import src.data_loader as dl
import src.feature_engineering as fe

from src.config import ALL_STATIONS, HORIZONS, SEQ_LEN_LSTM, RANDOM_SEED, get_station_paths
from src.feature_engineering import build_sequence_dataset, WEATHER_MODE_PERFECT_FULL
from src.models.hybrid_lstm import train_cnn_lstm, predict_cnn_lstm
from src.evaluation import compute_metrics
from src.utils import ensure_dirs, set_seed

set_seed(RANDOM_SEED)

WEATHER_MODE = WEATHER_MODE_PERFECT_FULL
# Override to run a subset, e.g. STATIONS_TO_RUN = ["MzWarChrosci"]
STATIONS_TO_RUN = ALL_STATIONS

print(f'Stations: {STATIONS_TO_RUN}')
print(f'Weather mode: {WEATHER_MODE}')

Stations: ['MzWarChrosci', 'MzOtwoBrzozo', 'MzWarWokalna', 'MzWarAlNiepo', 'MzLegZegrzyn', 'MzPiasPulask', 'MzWarBajkowa']
Weather mode: perfect_forecast_full


In [2]:
wall_start = time.time()
n = len(STATIONS_TO_RUN)

for i, station in enumerate(STATIONS_TO_RUN, 1):
    paths = get_station_paths(station)
    checkpoint = paths['results'] / 'C2_metrics.csv'

    if checkpoint.exists():
        print(f'[{i}/{n}] {station} — already done, skipping')
        continue

    print(f'\n[{i}/{n}] {station} — starting ...')
    t_station = time.time()

    ensure_dirs(paths['models'], paths['figures'], paths['results'])

    # Monkeypatch TARGET so src functions operate on the current station
    cfg.TARGET = station
    dl.TARGET = station
    fe.TARGET = station

    df = dl.load_data()
    train_df, test_df = dl.train_test_split(df)

    # ── Build sequence datasets ───────────────────────────────────────
    X_train_seq, y_train_seq, feature_names, scaler_X = build_sequence_dataset(
        train_df, seq_len=SEQ_LEN_LSTM, horizons=HORIZONS,
        fit_scaler=True, weather_mode=WEATHER_MODE
    )
    X_test_seq, y_test_seq, _, _ = build_sequence_dataset(
        test_df, seq_len=SEQ_LEN_LSTM, horizons=HORIZONS,
        scaler_X=scaler_X, fit_scaler=False, weather_mode=WEATHER_MODE
    )

    # 10% validation split from end of training sequences
    val_size = int(0.1 * len(X_train_seq))
    X_tr = X_train_seq[:-val_size]
    y_tr = y_train_seq[:-val_size]
    X_val = X_train_seq[-val_size:]
    y_val = y_train_seq[-val_size:]

    print(f'  Train: {X_tr.shape} | Val: {X_val.shape} | Test: {X_test_seq.shape}')

    # ── Train CNN-LSTM ────────────────────────────────────────────────
    model_path = paths['models'] / 'cnn_lstm_pfx_model.pt'
    model, history = train_cnn_lstm(
        X_tr, y_tr, X_val, y_val,
        epochs=100, batch_size=64, patience=15, lr=1e-3,
        save_path=model_path
    )

    # ── Evaluate on test set ─────────────────────────────────────────
    cnn_preds = predict_cnn_lstm(model, X_test_seq)
    y_test_true = np.expm1(y_test_seq)

    rows = []
    for h_idx, h in enumerate(HORIZONS):
        m = compute_metrics(y_test_true[:, h_idx], cnn_preds[:, h_idx])
        rows.append({'Model': 'C2_CNN_LSTM_pfxf', 'Station': station, 'Horizon': h, **m})

    # Save metrics immediately so checkpoint is valid on next restart
    pd.DataFrame(rows).to_csv(checkpoint, index=False)

    elapsed_min = (time.time() - t_station) / 60
    total_elapsed_min = (time.time() - wall_start) / 60
    avg_per_station = total_elapsed_min / i
    remaining_min = avg_per_station * (n - i)
    print(f'[{i}/{n}] {station} — done | elapsed {elapsed_min:.1f}min | est. remaining {remaining_min:.1f}min')

# Restore original TARGET
cfg.TARGET = 'MzWarChrosci'
dl.TARGET = 'MzWarChrosci'
fe.TARGET = 'MzWarChrosci'

print(f'\nAll stations complete in {(time.time() - wall_start) / 60:.1f}min total.')

11:18:45 | src.data_loader | INFO | Loading data from D:\MOJE\DATA_SCIENCE\ML_WARSAW_AQI_TOY\warsaw_aq_forecast\data\raw\FINAL_merged_PM25_1g_all_seasons.csv



[1/7] MzWarChrosci — starting ...


11:18:46 | src.data_loader | INFO | Missing values per column:
MzOtwoBrzozo    1084
MzWarWokalna    2028
MzWarAlNiepo     575
MzLegZegrzyn    1511
MzPiasPulask    1002
MzWarChrosci     416
MzWarBajkowa     636
dir_48            48
dir_24            48
11:18:46 | src.data_loader | INFO | Loaded 52608 rows, 27 columns, range 2019-01-01 00:00:00 → 2024-12-31 23:00:00
11:18:46 | src.data_loader | INFO | Train: 43824 rows (2019-01-01 00:00:00 → 2023-12-31 23:00:00)
11:18:46 | src.data_loader | INFO | Test : 8784 rows (2024-01-01 00:00:00 → 2024-12-31 23:00:00)
11:18:46 | src.feature_engineering | INFO | build_sequence_dataset | seq_len=48 | weather_mode=perfect_forecast_full
11:18:46 | src.feature_engineering | INFO | perfect_forecast_full: shifted 8 MET + 10 HYSPLIT_NUM + 2 HYSPLIT_CAT by -24 h
11:18:46 | src.feature_engineering | INFO | One-hot encoded HYSPLIT categoricals: ['dir_48', 'dir_24'] → 8 dummy cols
11:18:46 | src.feature_engineering | INFO | Retaining 10 HYSPLIT numeric columns

  Train: (34092, 48, 45) | Val: (3787, 48, 45) | Test: (7709, 48, 45)


11:18:52 | src.models.hybrid_lstm | INFO | CNN-LSTM | seq_len=48  n_features=45  n_horizons=24  device=cpu
11:18:52 | src.models.hybrid_lstm | INFO | Params: epochs=100  batch=64  patience=15  lr=0.00100
11:18:52 | src.models.hybrid_lstm | INFO | Training samples=34092  val samples=3787
11:18:52 | src.models.hybrid_lstm | INFO | -----------------------------------------------------------------
11:21:10 | src.models.hybrid_lstm | INFO | Epoch   1/100  train=0.17334  val=0.06122  best=0.06122  lr=1.00e-03 *
11:23:00 | src.models.hybrid_lstm | INFO | Epoch   2/100  train=0.05815  val=0.05670  best=0.05670  lr=1.00e-03 *
11:24:20 | src.models.hybrid_lstm | INFO | Epoch   3/100  train=0.04359  val=0.05925  best=0.05670  lr=1.00e-03  (no improve 1/15)
11:25:39 | src.models.hybrid_lstm | INFO | Epoch   4/100  train=0.03668  val=0.05576  best=0.05576  lr=1.00e-03 *
11:26:55 | src.models.hybrid_lstm | INFO | Epoch   5/100  train=0.03208  val=0.05768  best=0.05576  lr=1.00e-03  (no improve 1/15)

...

[6/7] MzPiasPulask — done | elapsed 22.4min | est. remaining 23.7min

[7/7] MzWarBajkowa — starting ...


13:41:02 | src.data_loader | INFO | Missing values per column:
MzOtwoBrzozo    1084
MzWarWokalna    2028
MzWarAlNiepo     575
MzLegZegrzyn    1511
MzPiasPulask    1002
MzWarChrosci     416
MzWarBajkowa     636
dir_48            48
dir_24            48
13:41:02 | src.data_loader | INFO | Loaded 52608 rows, 27 columns, range 2019-01-01 00:00:00 → 2024-12-31 23:00:00
13:41:02 | src.data_loader | INFO | Train: 43824 rows (2019-01-01 00:00:00 → 2023-12-31 23:00:00)
13:41:02 | src.data_loader | INFO | Test : 8784 rows (2024-01-01 00:00:00 → 2024-12-31 23:00:00)
13:41:02 | src.feature_engineering | INFO | build_sequence_dataset | seq_len=48 | weather_mode=perfect_forecast_full
13:41:02 | src.feature_engineering | INFO | perfect_forecast_full: shifted 8 MET + 10 HYSPLIT_NUM + 2 HYSPLIT_CAT by -24 h
13:41:02 | src.feature_engineering | INFO | One-hot encoded HYSPLIT categoricals: ['dir_48', 'dir_24'] → 8 dummy cols
13:41:02 | src.feature_engineering | INFO | Retaining 10 HYSPLIT numeric columns

  Train: (34424, 48, 46) | Val: (3824, 48, 46) | Test: (7786, 48, 46)


13:42:22 | src.models.hybrid_lstm | INFO | Epoch   1/100  train=0.19076  val=0.07025  best=0.07025  lr=1.00e-03 *
13:43:40 | src.models.hybrid_lstm | INFO | Epoch   2/100  train=0.06684  val=0.05635  best=0.05635  lr=1.00e-03 *
13:45:03 | src.models.hybrid_lstm | INFO | Epoch   3/100  train=0.05130  val=0.05519  best=0.05519  lr=1.00e-03 *
13:46:34 | src.models.hybrid_lstm | INFO | Epoch   4/100  train=0.04323  val=0.06147  best=0.05519  lr=1.00e-03  (no improve 1/15)
13:48:14 | src.models.hybrid_lstm | INFO | Epoch   5/100  train=0.03748  val=0.05440  best=0.05440  lr=1.00e-03 *
13:49:32 | src.models.hybrid_lstm | INFO | Epoch   6/100  train=0.03360  val=0.04937  best=0.04937  lr=1.00e-03 *
13:50:47 | src.models.hybrid_lstm | INFO | Epoch   7/100  train=0.03066  val=0.05783  best=0.04937  lr=1.00e-03  (no improve 1/15)
13:52:00 | src.models.hybrid_lstm | INFO | Epoch   8/100  train=0.02880  val=0.05455  best=0.04937  lr=1.00e-03  (no improve 2/15)
13:53:10 | src.models.hybrid_lstm | I

[7/7] MzWarBajkowa — done | elapsed 27.0min | est. remaining 0.0min

All stations complete in 169.3min total.
